In [0]:
from pyspark.sql.functions import *

def validate_silver_table(table_name, primary_key, surrogate_key):

    print("\n" + "="*80)
    print(f"Testing : {table_name}")
    print("="*80)

    table = f"{CATALOG}.{SCHEMA}.{table_name}"

    df = spark.table(table)

    # ---------------------------------------------------------
    # Test 1 : Table Exists
    # ---------------------------------------------------------
    print("\nTest 1 : Table Exists")

    assert spark.catalog.tableExists(table), "Table does not exist"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 2 : Row Count
    # ---------------------------------------------------------
    print("\nTest 2 : Row Count")

    rows = df.count()

    print("Rows :", rows)

    assert rows > 0, "Table is empty"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 3 : Duplicate Primary Key
    # ---------------------------------------------------------
    print("\nTest 3 : Duplicate Primary Key")

    duplicate = (
        df.groupBy(primary_key)
          .count()
          .filter("count>1")
          .count()
    )

    print("Duplicates :", duplicate)

    assert duplicate == 0, f"{duplicate} duplicate keys found"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 4 : Primary Key Null
    # ---------------------------------------------------------
    print("\nTest 4 : Primary Key Null")

    pk_null = df.filter(
        col(primary_key).isNull()
    ).count()

    print("Primary Key Nulls :", pk_null)

    assert pk_null == 0, f"{pk_null} NULL Primary Keys"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 5 : Surrogate Key Null
    # ---------------------------------------------------------
    print("\nTest 5 : Surrogate Key")

    sk_null = df.filter(
        col(surrogate_key).isNull()
    ).count()

    print("Surrogate Key Nulls :", sk_null)

    assert sk_null == 0, f"{sk_null} NULL Surrogate Keys"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 6 : Metadata
    # ---------------------------------------------------------
    print("\nTest 6 : Metadata")

    metadata = df.filter(

        col("ingestion_timestamp").isNull()

        | col("source_file").isNull()

        | col("batch_id").isNull()

        | col("load_date").isNull()

    ).count()

    print("Metadata Nulls :", metadata)

    assert metadata == 0, f"{metadata} Metadata NULLs"

    print("✅ Passed")


    # ---------------------------------------------------------
    # Test 7 : SCD Columns
    # ---------------------------------------------------------
    print("\nTest 7 : SCD Columns")

    current_null = df.filter(
        col("is_current").isNull()
    ).count()

    print("Current Flag Null :", current_null)

    assert current_null == 0, "Current Flag contains NULL"

    print("effective_from Null :", df.filter(col("effective_from").isNull()).count())

    print("effective_to Null :", df.filter(col("effective_to").isNull()).count())

    print("✅ Passed")


    print("\n🎉 ALL TESTS PASSED :", table_name)

In [0]:
CATALOG = "credit_risk_analysis_db"
SCHEMA = "silver_sch"

In [0]:
SILVER_TABLES = {

    "silver_applicant_profiles":{

        "primary_key":"applicant_id",

        "surrogate_key":"applicant_sk"

    },

    "silver_credit_application":{

        "primary_key":"applicant_id",

        "surrogate_key":"application_sk"

    },

    "silver_credit_history":{

        "primary_key":"applicant_id",

        "surrogate_key":"credit_history_sk"

    },

    "silver_loan_details":{

        "primary_key":"loan_id",

        "surrogate_key":"loan_sk"

    },

    "silver_economic_indicators":{

        "primary_key":"economic_sk",

        "surrogate_key":"economic_sk"

    }

}

In [0]:
for table, config in SILVER_TABLES.items():

    try:

        validate_silver_table(

            table,

            config["primary_key"],

            config["surrogate_key"]

        )

    except Exception as e:

        print("\n")
        print(" FAILED TABLE :", table)
        print("Reason :", e)

        break


Testing : silver_applicant_profiles

Test 1 : Table Exists
✅ Passed

Test 2 : Row Count
Rows : 148670
✅ Passed

Test 3 : Duplicate Primary Key
Duplicates : 0
✅ Passed

Test 4 : Primary Key Null
Primary Key Nulls : 0
✅ Passed

Test 5 : Surrogate Key
Surrogate Key Nulls : 0
✅ Passed

Test 6 : Metadata
Metadata Nulls : 0
✅ Passed

Test 7 : SCD Columns
Current Flag Null : 0
effective_from Null : 0
effective_to Null : 148670
✅ Passed

🎉 ALL TESTS PASSED : silver_applicant_profiles

Testing : silver_credit_application

Test 1 : Table Exists
✅ Passed

Test 2 : Row Count
Rows : 148670
✅ Passed

Test 3 : Duplicate Primary Key
Duplicates : 0
✅ Passed

Test 4 : Primary Key Null
Primary Key Nulls : 0
✅ Passed

Test 5 : Surrogate Key
Surrogate Key Nulls : 0
✅ Passed

Test 6 : Metadata
Metadata Nulls : 0
✅ Passed

Test 7 : SCD Columns
Current Flag Null : 0
effective_from Null : 0
effective_to Null : 148670
✅ Passed

🎉 ALL TESTS PASSED : silver_credit_application

Testing : silver_credit_history

Te